# Manufacturing Multi-Agent — Supervisor Agent

The supervisor routes incoming queries to the right sub-agent(s), collects results,
and synthesizes a technician-ready answer.

**Databricks AI Features:** Foundation Models, Vector Search, MLflow


Install LangChain packages first. `%pip` restarts Python, so it must run before `_config` sets up widgets and `sys.path`.


In [ ]:
%pip install databricks-langchain langchain-core -q


## Setup


In [ ]:
%run ./_config


Re-read job widgets in this notebook so catalog and schema are defined after `%run`.


In [ ]:
# Job parameters / widgets must be read in this notebook. %run does not always export names from .ipynb.
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
try:
    llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
except Exception:
    llm_endpoint = "databricks-claude-sonnet-4-5"
try:
    embedding_endpoint = dbutils.widgets.get("embedding_endpoint").strip()
except Exception:
    embedding_endpoint = "databricks-gte-large-en"
try:
    vs_endpoint = dbutils.widgets.get("vs_endpoint").strip()
except Exception:
    vs_endpoint = "mfg_quality_vs_endpoint"
try:
    eval_sample_size = dbutils.widgets.get("eval_sample_size").strip() or "50"
except Exception:
    eval_sample_size = "50"
try:
    cost_owner = dbutils.widgets.get("cost_owner").strip()
except Exception:
    cost_owner = ""
VS_INDEX_NAME = f"{catalog}.{schema}.service_bulletins_index"
print(f"Bound widgets: {catalog}.{schema}")


Load SQL guardrails, then the supervisor `run_agent` implementation.


Build the shared supervisor from `src/agent_runtime.py`. Spark SQL is the execute backend in notebooks.


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from agent_runtime import SUPERVISOR_SYSTEM, build_supervisor

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")
LLM_MODEL = llm_endpoint
VS_ENDPOINT_NAME = vs_endpoint

def execute_sql(sql_text):
    return spark.sql(sql_text).limit(20).toPandas().to_dict(orient="records")

llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0.1)
supervisor = build_supervisor(
    llm=llm,
    catalog=catalog,
    schema=schema,
    vs_index=VS_INDEX_NAME,
    execute_sql=execute_sql,
    workspace_client=WorkspaceClient(),
)
run_agent = supervisor.run
tools = supervisor.tools


## Test the Supervisor


Test 1: new defect report (classify + search).


In [ ]:
print("=" * 80)
print("TEST 1: New defect report")
print("=" * 80)
answer = run_agent(
    "I just found a brake rotor measuring 22.5mm on Line 2 at Arlington. "
    "That's way below the 28-32mm spec. The whole batch looks suspect. What should I do?"
)
print(f"\n{answer}")


Test 2: trend analysis (SQL).


In [ ]:
print("=" * 80)
print("TEST 2: Trend analysis")
print("=" * 80)
answer = run_agent(
    "What are the top 3 root causes for critical defects at Flint Assembly in the last 6 months, "
    "and what's the total rework cost?"
)
print(f"\n{answer}")


Test 3: hybrid question (multiple tools).


In [ ]:
print("=" * 80)
print("TEST 3: Hybrid — defect + history + bulletins")
print("=" * 80)
answer = run_agent(
    "We're seeing intermittent wiring harness failures on Line 3 at Fort Wayne. "
    "Is this a known issue? How many similar defects have we had recently, and what's the recommended fix?"
)
print(f"\n{answer}")


Test 4: plant KPI comparison.


In [ ]:
print("=" * 80)
print("TEST 4: Plant KPI comparison")
print("=" * 80)
answer = run_agent(
    "Compare the FTQ rates and OEE across all 5 plants for the last month. Which plant needs attention?"
)
print(f"\n{answer}")


## Log Supervisor Agent to MLflow


In [ ]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="mfg_quality_supervisor"):
    mlflow.log_params({
        "llm_model": LLM_MODEL,
        "vs_endpoint": VS_ENDPOINT_NAME,
        "vs_index": VS_INDEX_NAME,
        "num_tools": len(tools),
        "tool_names": ", ".join(t.name for t in tools),
        "max_iterations": 5,
    })
    mlflow.log_text(SUPERVISOR_SYSTEM, "supervisor_system_prompt.txt")
    tool_defs = {t.name: {"description": t.description, "args": str(t.args)} for t in tools}
    mlflow.log_dict(tool_defs, "tool_definitions.json")
    print("Supervisor agent logged to MLflow")
    print(f"   Run ID: {mlflow.active_run().info.run_id}")


## Summary


In [ ]:
print("Supervisor agent built and tested:")
print()
print("Architecture:")
print("  User Query → Supervisor (classification + routing)")
print("    - classify_defect (severity + root cause)")
print("    - search_service_bulletins (Vector Search RAG)")
print("    - query_quality_data (Text-to-SQL, read-only)")
print()
print("Next steps:")
print("  1. Optional: run 05_Eval_And_Cost.py")
print("  2. Open the Databricks App mfg-quality-assistant")
